In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

raw = (spark.read
       .option("header", "true")
       .option("inferSchema", "false")   # all strings: no lost digits in cc_num
       .csv("/Volumes/fintxn/ops/raw_source/fraudTrain.csv")
       .drop("_c0", "unix_time"))

print(raw.count())

In [0]:
CUSTOMER_COLS = ["cc_num", "first", "last", "gender", "street", "city",
                 "state", "zip", "lat", "long", "city_pop", "job", "dob"]

customers = raw.select(*CUSTOMER_COLS).dropDuplicates(["cc_num"])

# Stable surrogate key: C00001, C00002, ... ordered by card number
w = Window.orderBy("cc_num")
customers = customers.withColumn(
    "customer_id",
    F.format_string("C%05d", F.row_number().over(w)))

customers.write.mode("overwrite").saveAsTable("fintxn.ops.src_customers")
print(customers.count())

In [0]:
txns = (raw.join(customers.select("cc_num", "customer_id"), "cc_num")
           .select("trans_num", "trans_date_trans_time", "customer_id",
                   "merchant", "category", "amt",
                   "merch_lat", "merch_long", "is_fraud"))

txns.write.mode("overwrite").saveAsTable("fintxn.ops.src_transactions")
print(txns.count())

In [0]:
%sql
SELECT
  (SELECT count(*) FROM fintxn.ops.src_transactions)                  AS txn_rows,
  (SELECT count(DISTINCT trans_num) FROM fintxn.ops.src_transactions) AS distinct_trans_num,
  (SELECT count(*) FROM fintxn.ops.src_customers)                     AS customers,
  (SELECT count(DISTINCT cc_num) FROM fintxn.ops.src_customers)       AS distinct_cards,
  (SELECT min(trans_date_trans_time) FROM fintxn.ops.src_transactions) AS first_txn,
  (SELECT max(trans_date_trans_time) FROM fintxn.ops.src_transactions) AS last_txn,
  (SELECT round(avg(cast(is_fraud AS INT)), 4) FROM fintxn.ops.src_transactions) AS fraud_rate;